# Ares Trail — AI4MARS in Colab (works in VS Code too)
Run GPU work here, keep the web map in `index.html`. Same file runs in both places.
1. In Colab: Runtime > Change runtime type > T4 GPU. 2. In VS Code: just open this file and press Run All.

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
print('IN_COLAB =', IN_COLAB)
!nvidia-smi -L 2>&1 | head -n 3
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
# Kaggle auth: Colab Secrets (recommended) or upload kaggle.json
# Colab: put KAGGLE_USERNAME + KAGGLE_KEY in left panel Secrets, then:
import os
if IN_COLAB:
    try:
        from google.colab import userdata
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
        print('kaggle secrets loaded')
    except Exception as e:
        print('add KAGGLE_USERNAME/KAGGLE_KEY to Colab Secrets, or upload kaggle.json:', e)
!pip -q install kaggle pandas pillow numpy matplotlib

In [ ]:
# Download your 5 Kaggle sets (small subset first!)
import os
BASE = '/content/mars-data' if IN_COLAB else '../data/raw'
os.makedirs(BASE, exist_ok=True)
sets = {
 'ai4mars-2026': 'parthd144/ai4mars-dataset-june-2026',
 'crater-study': 'codebreaker619/mars-crater-study-dataset',
 'rems-weather': 'deepcontractor/mars-rover-environmental-monitoring-station',
}
for folder, slug in sets.items():
    out = f"{BASE}/{folder}"
    print('>>', slug)
    !kaggle datasets download -d {slug} -p {out} --unzip 2>&1 | tail -n 2

In [ ]:
# Quick look: AI4MARS classes + REMS weather strip
import glob, pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
imgs = glob.glob(f"{BASE}/ai4mars-2026/**/*.jpg", recursive=True)[:3]
print('ai4mars samples:', len(imgs))
for f in imgs:
    im = Image.open(f); print(f, im.size)
    plt.figure(); plt.imshow(im); plt.axis('off'); plt.title(f.split(chr(92))[-1][:40]); plt.show()
csvs = glob.glob(f"{BASE}/rems-weather/**/*.csv", recursive=True)
print('rems csvs:', csvs[:2])
if csvs:
    df = pd.read_csv(csvs[0]); print(df.head(3).to_string()); print(df.describe().to_string()

In [ ]:
# Export: terrain-penalty GeoJSON for index.html (replaces schematic AI blobs)
# Toy example: write 4 penalty circles -> ../data/processed/ai_terrain.geojson
import json, os
os.makedirs('../data/processed', exist_ok=True)
gj = {'type':'FeatureCollection','features':[
 {'type':'Feature','properties':{'cls':'bedrock','penalty':0.1},'geometry':{'type':'Point','coordinates':[77.43,18.465]}},
 {'type':'Feature','properties':{'cls':'sand','penalty':0.9},'geometry':{'type':'Point','coordinates':[77.50,18.445]}},
]}
open('../data/processed/ai_terrain.geojson','w').write(json.dumps(gj, indent=2))
print('wrote ai_terrain.geojson — load it in index.html gAI layer')

In [ ]:
# G4-1 LABELS: what the model must predict (AI4MARS classes -> traversal penalty)
# soil 0.2 | bedrock 0.1 | sand 0.9 | big-rock 1.0(blocked) | (rover/mast parts ignored)
# Official: crowdsourced MSL Navcam + M2020 images, merged-0.6 layout. Check label_keys.json after download.
import json, glob
keys = glob.glob(f"{BASE}/ai4mars-2026/**/label_keys.json", recursive=True)
print(keys[:2])
if keys:
    print(open(keys[0]).read()[:600])
PENALTY = {"soil":0.2,"bedrock":0.1,"sand":0.9,"big_rock":1.0,"big-rock":1.0}
print("penalty map:", PENALTY)

In [ ]:
# G4-2 TRAIN (subset, ~15 min on T4): fine-tune MobileNetV3-Small as image classifier
# Full segmentation (DeepLab) is Week-5 work - DROP for MVP. This classifier answers:
# "can terrain categories be recognized?" with measured accuracy (Eval Table 6).
import torch, torch.nn as nn
from torchvision import models, transforms
from PIL import Image
print("cuda:", torch.cuda.is_available())
w = models.MobileNet_V3_Small_Weights.DEFAULT
model = models.mobilenet_v3_small(weights=w)
model.classifier[3] = nn.Linear(model.classifier[3].in_features, 4)  # 4 terrain classes
tf = transforms.Compose([transforms.Resize((224,224)), transforms.ToTensor()])
# NOTE: wire real image->label pairs from ai4mars-2026 split here (see info.md/changelog.md).
# Subset demo: 200 train / 50 val. Record exact split in docs (Eval Table 6 row 1).
print("model ready. Next: fill train_loader/val_loader from dataset, run 3 epochs, save ai4mars_cls.pt")

In [ ]:
# G4-3 EVAL: accuracy + confusion matrix + failure montage (REQUIRED screenshots)
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
# y_true, y_pred = [...], [...]  # fill from val_loader after training
# print("acc:", (y_true==y_pred).mean())
# print(confusion_matrix(y_true, y_pred, labels=[0,1,2,3]))
# Rule: sand<->soil confusion is EXPECTED (dust-covered) - show 3 failure images, don't hide them.
# Pass bar for MVP: overall acc reported + per-class recall + 3 failures shown. No bar = SIMPLIFY to precomputed layer.
print("paste this cell's output figure + numbers into submission (Eval §21 item 4)")

In [ ]:
# G4-4 EXPORT: predictions -> penalty GeoJSON the map already loads (gAI layer)
# Heuristic until model lands: sand blobs 0.9 around dunes, bedrock 0.1 on delta.
# After training, replace radii/penalties with per-tile predicted class majority.
import json, os
os.makedirs("/content/export" if IN_COLAB else "../data/processed", exist_ok=True)
out = ("/content/export" if IN_COLAB else "../data/processed") + "/ai_terrain.geojson"
gj = {"type":"FeatureCollection","features":[
 {"type":"Feature","properties":{"cls":"bedrock","penalty":0.1,"src":"ai4mars-2026-cls"},
  "geometry":{"type":"Point","coordinates":[77.43,18.465]}},
 {"type":"Feature","properties":{"cls":"sand","penalty":0.9,"src":"ai4mars-2026-cls"},
  "geometry":{"type":"Point","coordinates":[77.50,18.445]}},
 {"type":"Feature","properties":{"cls":"big_rock","penalty":1.0,"src":"ai4mars-2026-cls"},
  "geometry":{"type":"Point","coordinates":[77.447,18.455]}},
]}
open(out,"w").write(json.dumps(gj, indent=2))
print("wrote", out, "-> copy to data/processed/ and the map's gAI + w2 use it live")